# Practice Session 04: Basket analysis

Association rule mining techniques are useful to analyze datasets consisting of transactions, in which each transaction is a collection of items.

We will use a well-known dataset named [Groceries](https://www.rdocumentation.org/packages/arules/versions/0.2-6/topics/Groceries) that is shipped with R as a test dataset. It contains 30 days of real-world data from a grocery shop, 9835 transactions in 169 product groups.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

⭐ **Regarding the usage of generative AI.** You may use generative AI to (a) understand how to do certain operations, and (b) generate short snippets of code, that you can incorporate into your practice after reviewing them. In that case, add a comment `#AI` before the snippet. You can also add `#AI (edited)` if you edited the code before incorporating it.

⚠️ It is forbidden to use generative AI in these practices for answering the text-based questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="#2680EB">Roger Trilla Piniés</font>

E-mail: <font color="#2680EB">roger.trilla01@estudiant.upf.edu</font>

AI used: <font color="#2680EB">Gemini, Google</font>

Date: <font color="#2680EB">19/10/2026</font>

In [3]:
import numpy as np  
import matplotlib.pyplot as plt  
import pandas as pd  
import csv
import gzip

import apyori
from apyori import apriori

If the apyori library is not already installed in your laptop, you can install it with: `%pip install apyori`, potentially adding the option `--break-system-packages`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 0. The Apriori Algorithm in a nutshell

There are three major components of Apriori algorithm, which we describe below using as an example the case where transactions are purchase histories.

**Support**: the number of transactions containing a particular item divided by total number of transactions:

   *Support(A) = (Transactions containing (A))/(Total Transactions)*

**Confidence**: normally indicates the likelihood that an item B is also bought if item A is bought. It can be calculated by finding the number of transactions where A and B are bought together, divided by total number of transactions where A is bought:

   *Confidence(A→B) = (Transactions containing both (A and B))/(Transactions containing A)*

**Lift**: the increase in the ratio of sale of B when A is sold. Lift(A –> B) can be calculated by dividing Confidence(A -> B) by Support(B):

   *Lift(A→B) = (Confidence (A→B))/(Support (B))*
   
A Lift of 1 means there is no association between products A and B. Lift greater than 1.0 means products A and B are more likely to be bought together. Lift less than 1.0 indicates two products are unlikely to be bought together.

The Apriori algorithm first finds itemsets having the desired level of support, and then within those itemsets tries to derive rules having the desired confidence and lift.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

# 1. Playing with apyori

The [apyori library](https://pypi.org/project/apyori/) is an implementation of the Apriori algorithm. Its typical usage is to receive a list of transactions and then print the association rules it found.

To use this library, we pass a list in which each element represents a transaction, for instance, imagine these are movies watched by people:

```python
transactions = [
    ['Sinners', 'Weapons', 'Companion'],
    ['Predator: Badlands', 'Bugonia', 'Companion', 'Frankenstein'],
    ['The Gorge', 'Sinners', 'Weapons', 'One Battle After Another'],
    ['Superman', 'F1', 'Predator: Badlands', 'Companion', 'Frankestein'],
    ['Thunderbolts', 'Superman', 'The Fantastic Four'],
    ['The Fantastic Four', 'Companion', 'Weapons'],
    ['Thunderbolts', 'The Fantastic Four', 'F1'],
    ['Weapons', 'Predator: Badlands', 'Frankestein', 'Companion'],
    ['Superman', 'Weapons', 'Frankestein', 'Black Bag'],
    ['Thunderbolds', 'Predator: Badlands', 'Companion', 'Frankestein'],
    ['Predator: Badlands', 'The Fantastic Four'],
    ['Frankestein', 'Companion', 'Sinners', 'Weapons'],
    ['The Gorge', 'One Battle After Another', 'Black Bag']   
]

results = list(apriori(transactions,
                       min_support=0.1,
                       min_confidence=0.9,
                       min_lift=1.0))
print_apyori_output(results)

```

The function `print_apyori_output`, which you can leave as-is, pretty-prints the output of the apyori library in a readable format.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [4]:
# LEAVE AS-IS

# Pretty-print apyori output
def print_apyori_output(association_rules):
    output = interpret_apyori_output(association_rules)

    for (itemset, rules) in output.items():
        print(f"Rules involving itemset {itemset}")
        for rule in rules:
            print(f"{rule["antecedent"]} => {rule["consequent"]}")
            print(f"  (sup:{rule["support"]:.4f}, conf:{rule["confidence"]:.2f}, lift:{rule["lift"]:.2f})")
    
        print()

# Returns a dictionary
#  The key is an itemset
#  The value is a list of rules involving that itemset
# Each rule is a dictionary, having
# antecedent, consequent, support, confidence, lift
def interpret_apyori_output(association_results):
    output = {}
    
    for relation_record in association_results:
        itemset = list(relation_record.items)
        
        # Consider only itemsets of two elements
        if len(itemset) > 1: 
        
            output[tuple(itemset)] = []
            support = relation_record.support

            for rules in relation_record.ordered_statistics:
                antecedent = list(rules.items_base)
                consequent = list(rules.items_add)
                           
                confidence = rules.confidence
                lift = rules.lift

                output[tuple(itemset)].append({
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "support": support,
                    "confidence": confidence,
                    "lift": lift,
                })
    return output

Next, invent your own set of transactions. Be **creative** but **tasteful,** and think of a list of transactions that **makes sense,** e.g., involving food, music, books, products, places, apps, or other items. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Provide example transactions**. Write at least 20 transactions.</font>

In [5]:

# AI, jo he escrit les primeres línies i, després he deixat que la AI ho acabés.
transactions = [
    ['Robert Downey Jr.', 'Chris Evans', 'Scarlett Johansson', 'Mark Ruffalo'],
    ['Robert Downey Jr.', 'Chris Evans', 'Samuel L. Jackson'],
    ['Cillian Murphy', 'Emily Blunt', 'Robert Downey Jr.', 'Matt Damon'],
    ['Timothée Chalamet', 'Zendaya', 'Florence Pugh', 'Javier Bardem'],
    ['Florence Pugh', 'Cillian Murphy', 'Emily Blunt'],
    ['Willem Dafoe', 'Tilda Swinton', 'Bill Murray', 'Scarlett Johansson'],
    ['Robert De Niro', 'Al Pacino', 'Leonardo DiCaprio'],
    ['Brad Pitt', 'Leonardo DiCaprio', 'Margot Robbie'],
    ['Ryan Gosling', 'Emma Stone', 'Steve Carell'],
    ['Daniel Radcliffe', 'Emma Watson', 'Rupert Grint'],
    ['Cillian Murphy', 'Emily Blunt', 'Florence Pugh', 'Matt Damon'],
    ['Robert Downey Jr.', 'Chris Evans', 'Scarlett Johansson', 'Samuel L. Jackson'],
    ['Timothée Chalamet', 'Zendaya', 'Florence Pugh'],
    ['Brad Pitt', 'Robert De Niro', 'Al Pacino'],
    ['Ryan Gosling', 'Emma Stone', 'Margot Robbie'],
    ['Matt Damon', 'Timothée Chalamet', 'Zendaya'],
    ['Scarlett Johansson', 'Florence Pugh', 'Samuel L. Jackson'],
    ['Cillian Murphy', 'Emily Blunt', 'Florence Pugh', 'Willem Dafoe'],
    ['Robert Downey Jr.', 'Chris Evans', 'Scarlett Johansson'],
    ['Florence Pugh', 'Timothée Chalamet', 'Emma Watson']
]

<font size="+1" color="#2680EB">**Run apriori over your transactions**. Execute the apriori algorithm, in which you should obtain at least **two** rules of the form ['A', 'B'] => ['C'], i.e., at least two rules having a 2-itemset in the antecedent and a 1-itemset in the consequent. Modify the transactions until you obtain such rules.</font>

In [6]:
results = list(apriori(transactions,
                       min_support=0.1,
                       min_confidence=0.9,
                       min_lift=1.0))
print_apyori_output(results)

Rules involving itemset ('Robert De Niro', 'Al Pacino')
['Al Pacino'] => ['Robert De Niro']
  (sup:0.1000, conf:1.00, lift:10.00)
['Robert De Niro'] => ['Al Pacino']
  (sup:0.1000, conf:1.00, lift:10.00)

Rules involving itemset ('Robert Downey Jr.', 'Chris Evans')
['Chris Evans'] => ['Robert Downey Jr.']
  (sup:0.2000, conf:1.00, lift:4.00)

Rules involving itemset ('Emily Blunt', 'Cillian Murphy')
['Cillian Murphy'] => ['Emily Blunt']
  (sup:0.2000, conf:1.00, lift:5.00)
['Emily Blunt'] => ['Cillian Murphy']
  (sup:0.2000, conf:1.00, lift:5.00)

Rules involving itemset ('Emma Stone', 'Ryan Gosling')
['Emma Stone'] => ['Ryan Gosling']
  (sup:0.1000, conf:1.00, lift:10.00)
['Ryan Gosling'] => ['Emma Stone']
  (sup:0.1000, conf:1.00, lift:10.00)

Rules involving itemset ('Zendaya', 'Timothée Chalamet')
['Zendaya'] => ['Timothée Chalamet']
  (sup:0.1500, conf:1.00, lift:5.00)

Rules involving itemset ('Robert Downey Jr.', 'Samuel L. Jackson', 'Chris Evans')
['Samuel L. Jackson', 'Chris E

<font size="+1" color="#2680EB">**Answer the questions below**.</font>

**Responses.**

**Describe briefly the set of transactions that you created, and how you created it:**

* He creat un conjunt de 20 transaccions reals que inclou diferents actors de gèneres diversos. La clau de les transaccions són els actors versàtils com podrien ser Florence Pugh, Scarlett Johansson o Robert Downey Jr., que actuen com a elements d'unió entre diferents gèneres i actors.

**Which are the two rules that you obtained?**

* Les dos regles que he aconseguit són que la parella Scarlett Johansson i Chris Evans => Robert Downey Jr. i, que la parella Matt Damon i Cillian Murphy => Emily Blunt.


**For one of those rules, can you provide the calculation of *support*?**

* Ho calcularé per la regla de Scarlett Johansson, Chris Evans i Robert Downey Jr.. Entre ells comparteixen 3 de les 20 transaccions totals. Per tant tenen un support del 3/20 = 15%.

**For the same rule, can you provide the calculation of *confidence*?**

* La confiança es calcula mirant les vegades que apareix l'antecedent i, les vegades que apareix tant l'antecedent com el conseqüent. Veiem que l'antecedent apareix el mateix nombre de vegades que apareix amb el conseqüent (3). No cal calcular-ho per saber que tenen una confiança del 100%.


# 2. Load grocery shopping dataset

The following code, which you should leave as-is, loads the transactions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [8]:
# LEAVE AS-IS

# File name
INPUT_TRANSACTIONS = "groceries.csv.gz"

# Load
MAX_TRANSACTIONS = 2000
transactions = []
with gzip.open(INPUT_TRANSACTIONS, "rt") as inputfile:
    reader = csv.reader(inputfile, delimiter=",")
    
    for item_list in reader:
        transactions.append(item_list)
        
        if len(transactions) >= MAX_TRANSACTIONS:
            break
        
               
print(f"Read {len(transactions)} transactions")

Read 2000 transactions


# 3. Extract association rules and comment on them

You are now ready to run the association rules mining algorithm over the selected transactions:

```python
results = list(apriori(transactions, min_support=..., min_confidence=..., min_lift=...))
print_apyori_output(results)
```

**Important:** if you set `min_support` to a very small value, your notebook will probably hang. Instead, define a `min_support_absolute` parameter and use `min_support=float(min_support_absolute)/num_transactions`. Start with a relatively high value of `min_support_absolute`

Start with a conservative value for the confidence and lift parameters, and increase it a bit if you find too many transactions satisfying those values.


<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Run apriori over these transactions**.</font>

In [9]:
num_transactions = len(transactions)

min_support_absolute = 30
min_sup = float(min_support_absolute) / num_transactions

# Executo l'algorisme Apriori
results = list(apriori(transactions, 
                       min_support=min_sup, 
                       min_confidence=0.3, 
                       min_lift=1.3))

print_apyori_output(results)

Rules involving itemset ('beef', 'other vegetables')
['beef'] => ['other vegetables']
  (sup:0.0195, conf:0.35, lift:1.80)

Rules involving itemset ('beef', 'rolls/buns')
['beef'] => ['rolls/buns']
  (sup:0.0215, conf:0.38, lift:1.86)

Rules involving itemset ('whole milk', 'beef')
['beef'] => ['whole milk']
  (sup:0.0225, conf:0.40, lift:1.54)

Rules involving itemset ('whole milk', 'berries')
['berries'] => ['whole milk']
  (sup:0.0160, conf:0.43, lift:1.67)

Rules involving itemset ('brown bread', 'other vegetables')
['brown bread'] => ['other vegetables']
  (sup:0.0205, conf:0.34, lift:1.78)

Rules involving itemset ('whole milk', 'brown bread')
['brown bread'] => ['whole milk']
  (sup:0.0255, conf:0.42, lift:1.64)

Rules involving itemset ('brown bread', 'yogurt')
['brown bread'] => ['yogurt']
  (sup:0.0180, conf:0.30, lift:2.14)

Rules involving itemset ('other vegetables', 'butter')
['butter'] => ['other vegetables']
  (sup:0.0190, conf:0.31, lift:1.64)

Rules involving itemset 

<font size="+1" color="#2680EB">**Comment on the transactions found**.</font>

**Responses.**

**Select one association rule that seems to describe something *causal*. Hint: consider someone buying ingredients for a recipe.**

* Margarine i rolls/buns, és una parella de compra que es complementa i és causal, ja que és pa amb mantega. Són dos aliments que s'acostumen a menjar junts i té sentit que es comprin a la vegada.

**Select one association rule that seems to be obtained merely because people tend to do trips to the supermarket to purchase several perishable, fast-expiration products.**

* Diria que other vegetables i whole milk, podria ser una compra que es fa al supermercat per rutina i tenir aliments frescos.

# 4. Re-implement computation of confidence and lift

The following code uses apyori to compute the support of an itemset. For efficiency, it computes the support of multiple itemsets.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [10]:
# LEAVE AS-IS

# Use apyori to obtain the number of transactions containing a list of itemsets
def compute_support_count(transactions, itemsets):
    num_transactions = len(transactions)
    
    # The instructor used AI to understand how to use apyori internals (Gemini 3.6 Flash)
    manager = apyori.TransactionManager.create(transactions)
    supports = []
    
    for itemset in itemsets:
        supports.append(int(num_transactions * manager.calc_support(frozenset(itemset))))
    return supports

def compute_support(transactions, itemsets):
    counts = compute_support_count(transactions, itemsets)
    return [count / len(transactions) for count in counts]

# Test this
print(compute_support_count(transactions, [["whole milk"], ["soda", "waffles"]]))
print(compute_support(transactions, [["whole milk"], ["soda", "waffles"]]))

[517, 24]
[0.2585, 0.012]


To check whether two numbers differ by at most *epsilon* from each other, you can use:

```python
assert abs(x - y) < epsilon
```

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Write code that computes support, confidence, and lift**. Check programmatically using assert that each of these is within 0.001 of the value returned by Apyori.</font>

In [11]:
def print_apyori_output_checking(association_rules):
    output = interpret_apyori_output(association_rules)

    for (itemset, rules) in output.items():
        print(f"Rules involving itemset {itemset}")
        for rule in rules:
            print(f"  apyori (sup:{rule['support']:.4f}, conf:{rule['confidence']:.2f}, lift:{rule['lift']:.2f})")

            # Elements de l'antecedent, conseqüent i conjunt global
            antecedent = rule["antecedent"]
            consequent = rule["consequent"]
            both = list(set(antecedent).union(set(consequent)))

            # Calculo els suports
            support = compute_support(transactions, [both])[0]
            support_ant = compute_support(transactions, [antecedent])[0]
            support_cons = compute_support(transactions, [consequent])[0]

            # Calculo la confiança i el lift
            confidence = support / support_ant if support_ant > 0 else 0.0
            lift = confidence / support_cons if support_cons > 0 else 0.0

            # Assert i validació
            epsilon = 0.001
            assert abs(support - rule["support"]) < epsilon, f"Support mismatch: {support} vs {rule['support']}"
            assert abs(confidence - rule["confidence"]) < epsilon, f"Confidence mismatch: {confidence} vs {rule['confidence']}"
            assert abs(lift - rule["lift"]) < epsilon, f"Lift mismatch: {lift} vs {rule['lift']}"

            print(f"  me (sup:{support:.4f}, conf:{confidence:.2f}, lift:{lift:.2f})")

        print()

print_apyori_output_checking(results)

Rules involving itemset ('beef', 'other vegetables')
  apyori (sup:0.0195, conf:0.35, lift:1.80)
  me (sup:0.0195, conf:0.35, lift:1.80)

Rules involving itemset ('beef', 'rolls/buns')
  apyori (sup:0.0215, conf:0.38, lift:1.86)
  me (sup:0.0215, conf:0.38, lift:1.86)

Rules involving itemset ('whole milk', 'beef')
  apyori (sup:0.0225, conf:0.40, lift:1.54)
  me (sup:0.0225, conf:0.40, lift:1.54)

Rules involving itemset ('whole milk', 'berries')
  apyori (sup:0.0160, conf:0.43, lift:1.67)
  me (sup:0.0160, conf:0.43, lift:1.67)

Rules involving itemset ('brown bread', 'other vegetables')
  apyori (sup:0.0205, conf:0.34, lift:1.78)
  me (sup:0.0205, conf:0.34, lift:1.78)

Rules involving itemset ('whole milk', 'brown bread')
  apyori (sup:0.0255, conf:0.42, lift:1.64)
  me (sup:0.0255, conf:0.42, lift:1.64)

Rules involving itemset ('brown bread', 'yogurt')
  apyori (sup:0.0180, conf:0.30, lift:2.14)
  me (sup:0.0180, conf:0.30, lift:2.14)

Rules involving itemset ('other vegetables',

# DELIVER (individually)

Remember to read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/data-mining-course/blob/master/upf/upf-evaluation.md).

Deliver a zip file containing:

* This notebook

## Extra points available

For more learning and extra points, find another dataset online (minimum 1000 transactions) and do association rule mining over that dataset.

**Note:** if you go for the extra points, add ``<font size="+2" color="#2680EB">Additional results: experiments on another dataset</font>`` at the top of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#2680EB">Additional results: experiments on another dataset</font>

In [21]:
# AI, m'ha ajudat a solucionar els errors que m'anaven sortint, ja que em sortia una divisió entre 0 i, no podia ser.

# Carreguem el dataset de la NBA
url = "https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv"
df_nba = pd.read_csv(url)

nba_transactions = []

# Filtro les temporades modernes del 2000 cap a l'actualitat. Només he agafat aquestes temporades ja que sinó era molt llarg.
df_modern = df_nba[df_nba['year_id'] >= 2000].copy()

for idx, row in df_modern.iterrows():
    tx = []
    
    # Punts anotats
    if row['pts'] >= 105:
        tx.append('High_Scoring (>105 pts)')
    elif row['pts'] <= 90:
        tx.append('Low_Scoring (<90 pts)')
        
    # Resultat del partit
    if row['game_result'] == 'W':
        tx.append('Win')
    elif row['game_result'] == 'L':
        tx.append('Loss')
        
    # Partit local o visitant
    if row['_iscopy'] == 0:
        tx.append('Home_Game')
    else:
        tx.append('Away_Game')
        
    # Nivell d'Elo (calcula lo bo que és l'equip)
    if row['elo_i'] >= 1600:
        tx.append('High_Elo_Team')
    elif row['elo_i'] <= 1400:
        tx.append('Low_Elo_Team')

    if len(tx) >= 2:
        nba_transactions.append(tx)

print(f"Total NBA transactions processed: {len(nba_transactions)}")

# Comprovació per evitar la divisió per zero
num_nba_tx = len(nba_transactions)

if num_nba_tx > 0:
    min_sup_abs_nba = 500  # Apareix com a mínim en 500 partits
    min_sup_nba = float(min_sup_abs_nba) / num_nba_tx

    nba_results = list(apriori(nba_transactions,
                               min_support=min_sup_nba,
                               min_confidence=0.5,
                               min_lift=1.1))

    print_apyori_output(nba_results)
else:
    print("No s'han generat transaccions.")

Total NBA transactions processed: 41092
Rules involving itemset ('Away_Game', 'Loss')
['Away_Game'] => ['Loss']
  (sup:0.3015, conf:0.60, lift:1.21)
['Loss'] => ['Away_Game']
  (sup:0.3015, conf:0.60, lift:1.21)

Rules involving itemset ('Low_Scoring (<90 pts)', 'Away_Game')
['Low_Scoring (<90 pts)'] => ['Away_Game']
  (sup:0.1627, conf:0.58, lift:1.16)

Rules involving itemset ('High_Elo_Team', 'Win')
['High_Elo_Team'] => ['Win']
  (sup:0.1473, conf:0.65, lift:1.31)

Rules involving itemset ('High_Scoring (>105 pts)', 'Home_Game')
['High_Scoring (>105 pts)'] => ['Home_Game']
  (sup:0.1630, conf:0.58, lift:1.16)

Rules involving itemset ('High_Scoring (>105 pts)', 'Win')
['High_Scoring (>105 pts)'] => ['Win']
  (sup:0.2153, conf:0.77, lift:1.54)

Rules involving itemset ('Home_Game', 'Win')
['Home_Game'] => ['Win']
  (sup:0.3015, conf:0.60, lift:1.21)
['Win'] => ['Home_Game']
  (sup:0.3015, conf:0.60, lift:1.21)

Rules involving itemset ('Low_Elo_Team', 'Loss')
['Low_Elo_Team'] => ['Lo

## 5. Extra Points: Association Rule Mining on an NBA Dataset

**Descripció del conjunt de dades:**
* El dataset  consta de 41.092 partits. Hi ha diferents categories en les quals podem analitzar els partits. El nivell de l'equip (ELO), la localització (Local/Visitant), el nivell d'anotació (< 90 o > 105) i el resultat final (Victòria/Derrota).

**Principals regles obtingudes:**
* La primera regla que he trobat és High Scoring, High Elo Team i Home Game => Win // Suport: 0.0422, Confiança: 0.90, Lift: 1.80

* La segona regla que he trobat és tot el contrari a la primera. Low Scoring, Away Game, Low Elo Team => Loss // Suport: 0.0324, Confiança: 0.92, Lift: 1.85

**Analisi de les regles:**
* De la primera podem dir que un 4,22% dels partits analitzats compleixen aquesta regla, que quan un equip que ha fet molts punts al partit, és un equip fort i juga a casa, guanya el 90% dels partits que juga. I es pot veure que té una forta associació  positiva (1.8), això fa que quan s'ajunten aquests antecedents la probabilitat de victòria puja molt.

* De la segona regla podem dir que el 3,24% dels partits analitzats compleixen la regla. Quan un equip amb poc nivell, que ha fet pocs punts i que juga com a visitant, el percentatge de derrota és del 92%. I, podem veure com el lift (1.85), encara és més alt que a la primera regla, per tant, quan passen aquests antecedents, la derrota és molt més possible que qualsevol altre partit.

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the AI instructions stated above.</font>